# Nigerian Commodity Price Volatility — Data Engineering Pipeline

**Period:** 2018–2025 | **Frequency:** Monthly Start (MS)

| Section | Source | Variables |
|---|---|---|
| 1 | FRED API | Brent Crude, US 10Y Yield |
| 2 | FEWS NET API | PMS (Gasoline), Diesel, Maize, Rice |
| 3 | Kaggle CSV | Official FX, Parallel FX, FX Premium, CPI, Food CPI |
| 4 | — | Cleaning: MAD filter, Spline impute, Log returns, Realized Vol |
| 5 | — | Master DataFrame assembly + export |

In [7]:
!pip install fredapi requests openpyxl -q

In [8]:
import pandas as pd
import numpy as np
import requests
import io
import os
from dotenv import load_dotenv
load_dotenv()
import warnings
from fredapi import Fred

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

# === CONFIGURATION ===
START_DATE = '2018-01-01'
END_DATE   = '2025-12-31'
FRED_API_KEY = os.getenv('FRED_API_KEY')

print('Imports OK.')

Imports OK.


---
## 1. Global Factors (FRED API)

In [9]:
def fetch_fred_data(api_key, start_date, end_date):
    fred = Fred(api_key=api_key)
    series_map = {
        'Brent_Crude_USD': 'POILBREUSDM',
        'US_10Y_Yield':    'GS10',
    }
    data = {}
    for name, sid in series_map.items():
        try:
            s = fred.get_series(sid, observation_start=start_date, observation_end=end_date)
            data[name] = s
            print(f'  ✓ {name} ({sid}): {len(s)} obs')
        except Exception as e:
            print(f'  ✗ {name}: {e}')
    df = pd.DataFrame(data)
    df.index = pd.to_datetime(df.index)
    df = df.resample('MS').mean()
    return df

print('Fetching FRED data...')
df_fred = fetch_fred_data(FRED_API_KEY, START_DATE, END_DATE)
print(f'\nFRED shape: {df_fred.shape}')
df_fred.head()

Fetching FRED data...
  ✓ Brent_Crude_USD (POILBREUSDM): 96 obs
  ✓ US_10Y_Yield (GS10): 96 obs

FRED shape: (96, 2)


,Brent_Crude_USD,US_10Y_Yield
2018-01-01,68.889130,2.58
2018-02-01,65.696500,2.86
2018-03-01,66.891818,2.84
2018-04-01,71.932857,2.87
2018-05-01,76.934348,2.98


---
## 2. Commodity Prices — FEWS NET API

Fetches **all 4 target commodities** from one source:
- **Gasoline** → PMS Price
- **Diesel** → Diesel Price
- **Maize** → Maize Price
- **Rice** → Rice Price

Aggregated to national monthly median across all markets.

In [10]:
def fetch_fewsnet_prices(start_date, end_date):
    """
    Fetch all 4 commodity prices from FEWS NET FDW API.
    Returns DataFrame with DatetimeIndex and price columns.
    """
    url = 'https://fdw.fews.net/api/marketpricefacts/'
    params = {'country': 'NG', 'format': 'csv'}

    # Map FEWS NET product names to our column names
    commodity_filters = {
        'PMS_Price':    'gasoline',
        'Diesel_Price': 'diesel',
        'Maize_Price':  'maize|corn',
        'Rice_Price':   'rice',
    }

    print('Fetching FEWS NET data (this may take a minute)...')
    try:
        resp = requests.get(url, params=params, timeout=120)
        if resp.status_code != 200:
            print(f'  ✗ HTTP {resp.status_code}')
            return pd.DataFrame()

        df = pd.read_csv(io.StringIO(resp.text))
        print(f'  Raw download: {len(df)} rows, {len(df.columns)} cols')

        # Known FEWS NET FDW columns
        commodity_col = 'product'
        date_col = 'period_date'
        price_col = 'value'

        for c in [commodity_col, date_col, price_col]:
            if c not in df.columns:
                print(f'  ✗ Column "{c}" not found. Available: {list(df.columns)}')
                return pd.DataFrame()

        print(f'  Commodities available: {sorted(df[commodity_col].dropna().unique())}')

        # Create lowercase lookup
        df['_cl'] = df[commodity_col].astype(str).str.lower()

        # Extract each commodity
        records = []
        for col_name, kw in commodity_filters.items():
            df_sub = df[df['_cl'].str.contains(kw, na=False)].copy()
            if df_sub.empty:
                print(f'  ⚠ No data for {col_name} (keyword: {kw})')
                continue

            df_sub[date_col] = pd.to_datetime(df_sub[date_col], errors='coerce')
            df_sub[price_col] = pd.to_numeric(df_sub[price_col], errors='coerce')
            df_sub = df_sub.dropna(subset=[date_col, price_col])

            monthly = df_sub.set_index(date_col).resample('MS')[price_col].median()
            monthly = monthly.loc[start_date:end_date].dropna()
            monthly.name = col_name
            records.append(monthly)

            if len(monthly) > 0:
                print(f'  ✓ {col_name}: {len(monthly)} months ({monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}), median ₦{monthly.median():,.0f}')
            else:
                print(f'  ⚠ {col_name}: no data in range')

        if records:
            df_out = pd.concat(records, axis=1)
            df_out.index = pd.to_datetime(df_out.index)
            return df_out
        else:
            return pd.DataFrame()

    except requests.exceptions.Timeout:
        print('  ✗ Timeout. Try again or download CSV from https://fews.net/data')
        return pd.DataFrame()
    except Exception as e:
        print(f'  ✗ Error: {e}')
        import traceback; traceback.print_exc()
        return pd.DataFrame()

df_commodities = fetch_fewsnet_prices(START_DATE, END_DATE)
print(f'\nCommodities shape: {df_commodities.shape}')
df_commodities.head()

Fetching FEWS NET data (this may take a minute)...
  Raw download: 82288 rows, 63 cols
  Commodities available: ['Bread', 'Cattle (Male)', 'Cowpeas (Brown)', 'Cowpeas (White)', 'Diesel', 'Gari (White)', 'Gari (Yellow)', 'Gasoline', 'Goats (Male)', 'Groundnuts (Shelled)', 'Maize Grain (White)', 'Maize Grain (Yellow)', 'Millet (Pearl)', 'Palm Oil (Refined)', 'Rice (5% Broken)', 'Rice (Milled)', 'Sheep (Male)', 'Sorghum (Brown)', 'Sorghum (White)', 'Yams']
  ✓ PMS_Price: 84 months (2018-01 to 2025-11), median ₦167
  ✓ Diesel_Price: 84 months (2018-01 to 2025-11), median ₦268
  ✓ Maize_Price: 84 months (2018-01 to 2025-11), median ₦7,152
  ✓ Rice_Price: 84 months (2018-01 to 2025-11), median ₦715

Commodities shape: (84, 4)


,PMS_Price,Diesel_Price,Maize_Price,Rice_Price
period_date,,,,
2018-01-01,197.400,211.00,4187.117603,6746.619718
2018-02-01,192.375,212.00,4045.684358,375.000000
2018-03-01,161.875,211.00,3947.974860,410.714286
2018-04-01,146.000,210.45,4425.265363,392.857143
2018-05-01,145.000,210.00,4276.641100,392.857143


In [11]:
# ==========================================
# RUN THIS CELL FIRST IF USING GOOGLE COLAB
# ==========================================
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print(" Drive mounted successfully.")

    !cp "/content/drive/MyDrive/monthly_exchange_cleaned.csv" ./
    print(" Copied monthly_exchange_cleaned.csv from Drive.")

    !cp "/content/drive/MyDrive/Money_and_Credit_Statistics_Data_in_Excel.xlsx" ./
    print(" Copied Money_and_Credit_Statistics_Data_in_Excel.xlsx from Drive.")

except ImportError:
    print("ℹ Not running in Google Colab, skipping drive mount.")
except Exception as e:
    print(f" Error: {e}")


Mounted at /content/drive
 Drive mounted successfully.
 Copied monthly_exchange_cleaned.csv from Drive.
 Copied Money_and_Credit_Statistics_Data_in_Excel.xlsx from Drive.


---
## 3. Domestic Macro — Kaggle CSV (FX + CPI)

Using **monthly_exchange_cleaned.csv** from Kaggle.  
Key columns: `ifem_dollar` (official FX), `bdc_dollar` (parallel FX), CPI inflation.

> ⚠ FX data ends ~April 2021. CPI is complete through 2024.

In [12]:
def load_kaggle_macro(csv_path, start_date, end_date):
    """
    Load and process the Kaggle monthly_exchange_cleaned.csv.
    Maps known columns to standardized names.
    """
    if not os.path.exists(csv_path):
        print(f'⚠ File not found: {csv_path}')
        print('  Upload monthly_exchange_cleaned.csv to Colab:')
        print('  1. Put file in Google Drive')
        print('  2. Run: from google.colab import drive; drive.mount("/content/drive")')
        print('  3. !cp /content/drive/MyDrive/monthly_exchange_cleaned.csv data/')
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    print(f'Loaded {len(df)} rows.')

    # Parse date
    df['Date'] = pd.to_datetime(df['date'])
    df = df.set_index('Date').sort_index()
    df = df.loc[start_date:end_date]

    # Map to standardized names
    col_map = {
        'ifem_dollar':       'Official_FX',
        'bdc_dollar':        'Parallel_FX',
        'allitems_year_on':  'CPI_YoY',
        'food_year_on':      'Food_CPI_YoY',
        'cpi_monthly':       'CPI_Monthly',
    }

    mapped = {}
    for orig, new in col_map.items():
        if orig in df.columns:
            mapped[new] = pd.to_numeric(df[orig], errors='coerce')

    df_out = pd.DataFrame(mapped, index=df.index)

    # Replace 0 with NaN for FX (0 means missing, not free)
    for fx_col in ['Official_FX', 'Parallel_FX']:
        if fx_col in df_out.columns:
            df_out[fx_col] = df_out[fx_col].replace(0, np.nan)

    # Compute FX Premium
    if 'Parallel_FX' in df_out.columns and 'Official_FX' in df_out.columns:
        df_out['FX_Premium'] = (df_out['Parallel_FX'] / df_out['Official_FX']) - 1

    # Resample to Monthly Start
    df_out = df_out.resample('MS').mean()

    # Report completeness
    print(f'\nColumns mapped: {list(df_out.columns)}')
    print(f'\nCompleteness (2018-2025):')
    completeness = (1 - df_out.isna().mean()) * 100
    for col in df_out.columns:
        print(f'  {col}: {completeness[col]:.0f}%')

    return df_out

# Try common paths
for path in ['monthly_exchange_cleaned.csv', 'data/monthly_exchange_cleaned.csv', 'data/nigeria_macro_fx.csv']:
    if os.path.exists(path):
        CBN_CSV_PATH = path
        break
else:
    CBN_CSV_PATH = 'data/monthly_exchange_cleaned.csv'

df_macro = load_kaggle_macro(CBN_CSV_PATH, START_DATE, END_DATE)
print(f'\nMacro shape: {df_macro.shape}')
df_macro.head()

Loaded 420 rows.

Columns mapped: ['Official_FX', 'Parallel_FX', 'CPI_YoY', 'Food_CPI_YoY', 'CPI_Monthly', 'FX_Premium']

Completeness (2018-2025):
  Official_FX: 48%
  Parallel_FX: 48%
  CPI_YoY: 100%
  Food_CPI_YoY: 100%
  CPI_Monthly: 100%
  FX_Premium: 48%

Macro shape: (84, 6)


,Official_FX,Parallel_FX,CPI_YoY,Food_CPI_YoY,CPI_Monthly,FX_Premium
Date,,,,,,
2018-01-01,305.78,363.20,15.13,18.92,16.22,0.187782
2018-02-01,305.90,362.48,14.33,17.59,15.93,0.184962
2018-03-01,305.74,362.07,13.34,16.08,15.60,0.184242
2018-04-01,305.61,362.25,12.48,14.80,15.20,0.185334
2018-05-01,305.83,362.86,11.61,13.45,14.79,0.186476


---
## 4. CBN Manual Upload (M2 Data)

Loads the Money Supply (M2) from the official CBN Excel file (`Money_and_Credit_Statistics_Data_in_Excel.xlsx`).

In [13]:
# ===============================================
# 4. CBN Manual Upload (M2 portion)
# ===============================================
import pandas as pd
import os

def load_cbn_m2(file_path, start_date, end_date):
    """Load M2 data from CBN Money and Credit Statistics Excel file."""
    if not os.path.exists(file_path):
        print(f"⚠ M2 file not found: {file_path}")
        return pd.DataFrame()

    # Load the data, using the first row as header (as detected previously)
    df = pd.read_excel(file_path, header=0)
    print(f"Loaded CBN M2 Data: {len(df)} rows.")

    # Create the Date column from period (e.g. "January 2026")
    df["Date"] = pd.to_datetime(df["period"], format="%B %Y", errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()

    # Extract only M2
    df = df[["moneySupply_M2"]].rename(columns={"moneySupply_M2": "M2_NGN"})

    # Convert to numeric, handle any weird string values
    df["M2_NGN"] = pd.to_numeric(df["M2_NGN"], errors="coerce")

    # Filter to our study period and resample to Monthly Start explicitly
    df = df.loc[start_date:end_date]
    df = df.resample("MS").mean()

    return df

# Look in both the local directory and mounted drive
possible_paths = [
    "Money_and_Credit_Statistics_Data_in_Excel.xlsx",
    "/content/drive/MyDrive/Money_and_Credit_Statistics_Data_in_Excel.xlsx"
]

CBN_M2_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        CBN_M2_PATH = p
        break

if CBN_M2_PATH:
    df_cbn_manual = load_cbn_m2(CBN_M2_PATH, START_DATE, END_DATE)
    print(f"\n CBN M2 shape: {df_cbn_manual.shape}")
    display(df_cbn_manual.head(3))
else:
    df_cbn_manual = pd.DataFrame()
    print(" CBN M2 file not found. Ensure it is uploaded to Colab.")


Loaded CBN M2 Data: 430 rows.

 CBN M2 shape: (96, 1)


,M2_NGN
Date,
2018-01-01,23963031.48
2018-02-01,24143010.28
2018-03-01,24424422.14


---
## 5. Data Cleaning Pipeline

1. **MAD Z-score filter** ($|Z_t| > 3.5$) — outliers → NaN
2. **Cubic Spline interpolation** (order=3) — fill gaps
3. **Targets:** log returns + 12-month annualized realized volatility

In [14]:
def mad_zscore_filter(series, threshold=3.5):
    s = series.copy()
    median = s.median()
    mad = np.median(np.abs(s - median))
    if mad == 0:
        return s
    mad_zscore = 0.6745 * (s - median) / mad
    n_outliers = (np.abs(mad_zscore) > threshold).sum()
    if n_outliers > 0:
        print(f'    MAD: {n_outliers} outliers in {series.name}')
    s[np.abs(mad_zscore) > threshold] = np.nan
    return s

def spline_impute(series, order=3):
    n_valid = series.dropna().shape[0]
    if n_valid < order + 1:
        return series.interpolate(method='linear')
    try:
        return series.interpolate(method='spline', order=order)
    except:
        return series.interpolate(method='linear')

def compute_targets(price_series, window=12):
    log_returns = np.log(price_series / price_series.shift(1))
    realized_vol = log_returns.rolling(window=window).std() * np.sqrt(12)
    return log_returns, realized_vol

print('Cleaning functions defined.')

Cleaning functions defined.


---
## 6. Master DataFrame Assembly

In [15]:
# === MERGE ALL SOURCES ===
print('=== Merging DataFrames ===')

df_master = df_fred.copy()
print(f'  FRED:        {df_fred.shape}')

# Commodities (FEWS NET): PMS, Diesel, Maize, Rice
if 'df_commodities' in dir() and isinstance(df_commodities, pd.DataFrame) and not df_commodities.empty:
    df_commodities.index = pd.to_datetime(df_commodities.index)
    df_master = df_master.join(df_commodities, how='outer')
    print(f'  + Commodities: {df_commodities.shape}')
else:
    print(f'  - Commodities: skipped')

# Macro (Kaggle): FX, CPI
if 'df_macro' in dir() and isinstance(df_macro, pd.DataFrame) and not df_macro.empty:
    df_macro.index = pd.to_datetime(df_macro.index)
    df_master = df_master.join(df_macro, how='outer')
    print(f"  + Macro:       {df_macro.shape}")
else:
    print(f"  - Macro: skipped")

# CBN manual (M2)
if "df_cbn_manual" in dir() and isinstance(df_cbn_manual, pd.DataFrame) and not df_cbn_manual.empty:
    df_cbn_manual.index = pd.to_datetime(df_cbn_manual.index)
    df_master = df_master.join(df_cbn_manual, how="outer")
    print(f"  + CBN M2:      {df_cbn_manual.shape}")
else:
    print(f"  - CBN M2: skipped (not loaded)")

# Ensure DatetimeIndex and filter
df_master.index = pd.to_datetime(df_master.index)
df_master = df_master.sort_index()
df_master = df_master.loc[START_DATE:END_DATE]

print(f'\nMerged shape: {df_master.shape}')
print(f'Date range: {df_master.index.min()} to {df_master.index.max()}')
print(f'\nMissing values:\n{df_master.isna().sum()}')

=== Merging DataFrames ===
  FRED:        (96, 2)
  + Commodities: (84, 4)
  + Macro:       (84, 6)
  + CBN M2:      (96, 1)

Merged shape: (96, 13)
Date range: 2018-01-01 00:00:00 to 2025-12-01 00:00:00

Missing values:
Brent_Crude_USD     0
US_10Y_Yield        0
PMS_Price          12
Diesel_Price       12
Maize_Price        12
Rice_Price         12
Official_FX        56
Parallel_FX        56
CPI_YoY            12
Food_CPI_YoY       12
CPI_Monthly        12
FX_Premium         56
M2_NGN              0
dtype: int64


In [16]:
# === APPLY CLEANING PIPELINE ===
print('=== Cleaning Pipeline ===')

price_cols = [c for c in df_master.columns if df_master[c].dtype in ['float64', 'int64']]
print(f'Columns to clean: {price_cols}\n')

print('Step 1: MAD Z-score filter (|Z| > 3.5)...')
for col in price_cols:
    df_master[col] = mad_zscore_filter(df_master[col], threshold=3.5)

print('\nStep 2: Cubic spline interpolation...')
for col in price_cols:
    n_before = df_master[col].isna().sum()
    df_master[col] = spline_impute(df_master[col], order=3)
    n_after = df_master[col].isna().sum()
    filled = n_before - n_after
    if filled > 0:
        print(f'    {col}: filled {filled} gaps')

print(f'\nRemaining NaN:\n{df_master.isna().sum()}')

=== Cleaning Pipeline ===
Columns to clean: ['Brent_Crude_USD', 'US_10Y_Yield', 'PMS_Price', 'Diesel_Price', 'Maize_Price', 'Rice_Price', 'Official_FX', 'Parallel_FX', 'CPI_YoY', 'Food_CPI_YoY', 'CPI_Monthly', 'FX_Premium', 'M2_NGN']

Step 1: MAD Z-score filter (|Z| > 3.5)...

Step 2: Cubic spline interpolation...
    PMS_Price: filled 12 gaps
    Diesel_Price: filled 12 gaps
    Maize_Price: filled 12 gaps
    Rice_Price: filled 12 gaps
    Official_FX: filled 56 gaps
    Parallel_FX: filled 56 gaps
    CPI_YoY: filled 12 gaps
    Food_CPI_YoY: filled 12 gaps
    CPI_Monthly: filled 12 gaps
    FX_Premium: filled 56 gaps

Remaining NaN:
Brent_Crude_USD    0
US_10Y_Yield       0
PMS_Price          0
Diesel_Price       0
Maize_Price        0
Rice_Price         0
Official_FX        0
Parallel_FX        0
CPI_YoY            0
Food_CPI_YoY       0
CPI_Monthly        0
FX_Premium         0
M2_NGN             0
dtype: int64


In [17]:
# === COMPUTE TARGET VARIABLES ===
print('=== Computing Targets ===')

commodity_cols = [c for c in ['PMS_Price', 'Diesel_Price', 'Maize_Price', 'Rice_Price']
                  if c in df_master.columns]

for col in commodity_cols:
    base = col.replace('_Price', '')
    log_ret, real_vol = compute_targets(df_master[col], window=12)
    df_master[f'{base}_LogReturn'] = log_ret
    df_master[f'{base}_RealVol'] = real_vol
    print(f'  ✓ {base}: LogReturn + RealVol')

print(f'\nFinal shape: {df_master.shape}')
print(f'Columns: {list(df_master.columns)}')

=== Computing Targets ===
  ✓ PMS: LogReturn + RealVol
  ✓ Diesel: LogReturn + RealVol
  ✓ Maize: LogReturn + RealVol
  ✓ Rice: LogReturn + RealVol

Final shape: (96, 21)
Columns: ['Brent_Crude_USD', 'US_10Y_Yield', 'PMS_Price', 'Diesel_Price', 'Maize_Price', 'Rice_Price', 'Official_FX', 'Parallel_FX', 'CPI_YoY', 'Food_CPI_YoY', 'CPI_Monthly', 'FX_Premium', 'M2_NGN', 'PMS_LogReturn', 'PMS_RealVol', 'Diesel_LogReturn', 'Diesel_RealVol', 'Maize_LogReturn', 'Maize_RealVol', 'Rice_LogReturn', 'Rice_RealVol']


In [18]:
# === SAVE ===
os.makedirs('output', exist_ok=True)
df_master.to_csv('output/master_dataset.csv')
print(f'✓ Saved to output/master_dataset.csv')
print(f'  Shape: {df_master.shape}')
print(f'  Period: {df_master.index.min():%Y-%m} to {df_master.index.max():%Y-%m}')
print(f'\n=== Summary ===')
df_master.describe().round(2)

✓ Saved to output/master_dataset.csv
  Shape: (96, 21)
  Period: 2018-01 to 2025-12

=== Summary ===


,Brent_Crude_USD,US_10Y_Yield,PMS_Price,Diesel_Price,Maize_Price,Rice_Price,Official_FX,Parallel_FX,CPI_YoY,Food_CPI_YoY,CPI_Monthly,FX_Premium,M2_NGN,PMS_LogReturn,PMS_RealVol,Diesel_LogReturn,Diesel_RealVol,Maize_LogReturn,Maize_RealVol,Rice_LogReturn,Rice_RealVol
count,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,9.600000e+01,95.00,84.00,95.00,84.00,94.00,83.00,95.00,84.00
mean,72.44,2.85,388.52,632.48,17734.77,2860.70,-6102.96,-17473.76,-211.90,-73.10,19.91,0.65,5.726631e+07,0.02,0.29,0.02,0.19,0.01,2.49,0.01,2.15
std,16.65,1.26,330.18,446.93,33724.66,5118.76,10721.28,29930.28,874.40,349.88,8.73,0.57,3.334461e+07,0.11,0.24,0.07,0.13,0.89,2.15,0.82,1.72
min,26.85,0.62,124.65,192.67,-90535.18,364.51,-40740.60,-114561.92,-5526.85,-2150.28,11.27,0.15,2.396303e+07,-0.20,0.01,-0.15,0.02,-3.46,0.19,-2.89,0.07
25%,64.06,1.68,145.00,230.00,3523.80,605.41,-8191.68,-23047.45,11.36,13.51,12.75,0.18,2.904508e+07,-0.01,0.11,-0.02,0.08,-0.04,0.54,-0.01,0.26
50%,73.37,2.90,184.31,349.00,7962.05,765.13,120.29,-155.13,15.42,17.38,16.66,0.38,4.459120e+07,0.00,0.24,0.00,0.16,0.01,1.06,0.02,2.57
75%,81.89,4.07,683.04,1112.20,17554.86,2144.70,306.95,362.31,21.37,23.84,24.83,0.97,8.210968e+07,0.03,0.37,0.03,0.31,0.13,4.91,0.10,3.73
max,117.69,4.80,1175.33,1531.25,150195.35,24549.53,381.00,481.60,34.60,40.87,41.11,2.15,1.244002e+08,0.74,0.80,0.41,0.43,3.28,5.95,2.85,5.07


In [19]:
# === DIAGNOSTICS ===
print('=== Data Quality ===\n')

# Completeness
completeness = (1 - df_master.isna().mean()) * 100
print('Completeness (%):'); print(completeness.round(1).to_string())

# Date continuity
expected = pd.date_range(df_master.index.min(), df_master.index.max(), freq='MS')
missing = expected.difference(df_master.index)
print(f'\nDate gaps: {len(missing)} missing months out of {len(expected)}')

# Structural break — May 2023
break_date = '2023-05-01'
if break_date in df_master.index.strftime('%Y-%m-%d').tolist():
    print(f'\nStructural break (May 2023 subsidy removal):')
    for col in commodity_cols:
        if col in df_master.columns:
            pre  = df_master.loc[:break_date, col].iloc[-6:].mean()
            post = df_master.loc[break_date:, col].iloc[:6].mean()
            chg  = (post / pre - 1) * 100 if pre > 0 else float('nan')
            print(f'  {col}: pre=₦{pre:,.0f}, post=₦{post:,.0f}, change={chg:+.1f}%')

print('\n✓ Pipeline complete.')

=== Data Quality ===

Completeness (%):
Brent_Crude_USD     100.0
US_10Y_Yield        100.0
PMS_Price           100.0
Diesel_Price        100.0
Maize_Price         100.0
Rice_Price          100.0
Official_FX         100.0
Parallel_FX         100.0
CPI_YoY             100.0
Food_CPI_YoY        100.0
CPI_Monthly         100.0
FX_Premium          100.0
M2_NGN              100.0
PMS_LogReturn        99.0
PMS_RealVol          87.5
Diesel_LogReturn     99.0
Diesel_RealVol       87.5
Maize_LogReturn      97.9
Maize_RealVol        86.5
Rice_LogReturn       99.0
Rice_RealVol         87.5

Date gaps: 0 missing months out of 96

Structural break (May 2023 subsidy removal):
  PMS_Price: pre=₦295, post=₦544, change=+84.3%
  Diesel_Price: pre=₦874, post=₦805, change=-7.9%
  Maize_Price: pre=₦8,697, post=₦10,701, change=+23.0%
  Rice_Price: pre=₦989, post=₦1,072, change=+8.4%

✓ Pipeline complete.
